<a href="https://colab.research.google.com/github/Melonangie/Galaxy-Vision/blob/main/extract_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Spectral feature extraction for MaNGA 1D spectra  ->  morphology table

We extracted 14 standard spectral features from WAVE and FLUX_1RE.

Together they describe a galaxy's stellar age, metal content and star formation, and these properties track its shape. Early-type galaxies (E/S0) tend to have strong D4000 and Mgb values and weak emission lines. Late-type spirals tend to have strong Hδ absorption and strong Hα and [OII] emission.

| # | Column | What it measures | Typical: early types → late types | Source of the definition |
|---|---|---|---|---|
| 1 | `D4000n` | 4000 Å break (age of the stars) | high (~1.9) → low (~1.2) | Balogh et al. 1999, ApJ 527, 54 |
| 2 | `HdA` | Hδ absorption (young A-type stars) | low → high | Worthey & Ottaviani 1997, ApJS 111, 377 |
| 3 | `Hb_abs` | Lick Hβ absorption | low → high | Worthey et al. 1994, ApJS 94, 687; Trager et al. 1998, ApJS 116, 1 |
| 4 | `Mgb` | Mg absorption (metals, α-elements) | high → low | Worthey et al. 1994; Trager et al. 1998 |
| 5 | `Fe5270` | Fe absorption | high → low | Worthey et al. 1994; Trager et al. 1998 |
| 6 | `Fe5335` | Fe absorption | high → low | Worthey et al. 1994; Trager et al. 1998 |
| 7 | `MgFe_prime` | Overall metal content, combining Mgb and Fe | high → low | Thomas, Maraston & Bender 2003, MNRAS 339, 897 |
| 8 | `NaD` | Na absorption (stars and dust) | high → low | Worthey et al. 1994; Trager et al. 1998 |
| 9 | `EW_OII` | [OII] 3727 emission | ~0 → high | Custom windows; line centre from the SDSS line list |
| 10 | `EW_Hb_em` | Hβ emission | ~0 → high | Custom windows; line centre from the SDSS line list |
| 11 | `EW_OIII` | [OIII] 5007 emission | ~0 → variable | Custom windows; line centre from the SDSS line list |
| 12 | `EW_Ha` | Hα emission (star formation) | ~0 → high | Custom windows; line centre from the SDSS line list |
| 13 | `EW_NII` | [NII] 6584 emission | low → medium | Custom windows; line centre from the SDSS line list |
| 14 | `log_N2Ha` | log([NII]/Hα), a rough ionization diagnostic | high → low | Calculated from `EW_NII` and `EW_Ha` |

**Notes**
- Rest-frame conversion: (`λ_rest = λ_obs / (1 + nsa_z)`), using nsa_z from the morphology table. Galaxies with nsa_z = -9999 get NaN for every feature.
- Vacuum vs. air wavelengths: MaNGA wavelengths are in vacuum, but the Lick and Balogh bands are defined in air. The code converts using Morton (1991).
- Gaps in the spectrum: A feature is only measured if every band it needs is at least 80% covered. the galaxy has no valid `nsa_z`. Otherwise it is NaN.
- Missing values: the morphology table already has NaN values. Use a model that tolerates them, such as HistGradientBoostingClassifier, or impute them before training.


In [1]:
%pip -q install astropy tqdm

In [8]:
import glob, os, re
import numpy as np
from astropy.table import Table
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Proyecto_morfologia/data/morfologia.fits" /content/
!cp "/content/drive/MyDrive/Proyecto_morfologia/data/MPL-11.zip" /content/
!unzip -q /content/MPL-11.zip -d /content/fits
!df -h /content
!ls /content

Mounted at /content/drive
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   23G   85G  22% /
drive  fits  morfologia.fits  MPL-11.zip  sample_data


 0. Setup

In [3]:
SPECTRA_DIR   = "/content/fits"            # folder with the 10,243 *.fits.gz
MORPH_FILE    = "/content/morfologia.fits"     # table like morfologia-sample.csv
OUTPUT_FILE   = "/content/morfologia_features.fits"
MIN_COVERAGE  = 0.8   # a band needs >=80% of its width covered by pixels

1. Band definitions (REST-FRAME, AIR wavelengths, in Angstrom)

Format: (blue continuum), (feature band), (red continuum)

In [4]:
# Lick/IDS absorption indices
#   Source: Worthey, Faber, Gonzalez & Burstein 1994, ApJS 94, 687
#           Trager, Worthey, Faber, Burstein & Gonzalez 1998, ApJS 116, 1 (Table 2)
#   HdA source: Worthey & Ottaviani 1997, ApJS 111, 377 (Table 1)
LICK_BANDS = {
    "HdA":    ((4041.600, 4079.750), (4083.500, 4122.250), (4128.500, 4161.000)),
    "Hb_abs": ((4827.875, 4847.875), (4847.875, 4876.625), (4876.625, 4891.625)),
    "Mgb":    ((5142.625, 5161.375), (5160.125, 5192.625), (5191.375, 5206.375)),
    "Fe5270": ((5233.150, 5248.150), (5245.650, 5285.650), (5285.650, 5318.150)),
    "Fe5335": ((5304.625, 5315.875), (5312.125, 5352.125), (5353.375, 5363.375)),
    "NaD":    ((5860.625, 5875.625), (5876.875, 5909.375), (5922.125, 5948.125)),
}

# Emission-line equivalent widths.
#   Line centres: SDSS line list (https://classic.sdss.org/dr6/algorithms/linestable.php)
#   The windows below are SIMPLE windows defined for this script (Lick-style,
#   ~20 A wide, continuum chosen to avoid neighbouring lines). They are NOT a
#   published standard -- they are good enough as ML features.
EMISSION_BANDS = {
    "EW_OII":   ((3696.0, 3716.0), (3716.0, 3740.0), (3740.0, 3760.0)),  # [OII] 3726+3729
    "EW_Hb_em": ((4827.9, 4847.9), (4851.0, 4871.0), (4876.6, 4891.6)),  # Hβ 4861
    "EW_OIII":  ((4977.0, 4995.0), (4997.0, 5017.0), (5022.0, 5042.0)),  # [OIII] 5007
    "EW_Ha":    ((6483.0, 6513.0), (6553.0, 6573.0), (6623.0, 6653.0)),  # Hα 6563
    "EW_NII":   ((6483.0, 6513.0), (6575.0, 6593.0), (6623.0, 6653.0)),  # [NII] 6584
}

# 4000 A break, narrow definition
#   Source: Balogh, Morris, Yee, Carlberg & Ellingson 1999, ApJ 527, 54
D4000_BLUE = (3850.0, 3950.0)
D4000_RED  = (4000.0, 4100.0)

FEATURE_NAMES = (["D4000n"] + list(LICK_BANDS) + ["MgFe_prime"] + list(EMISSION_BANDS) + ["log_N2Ha"])



2. Helper functions

In [5]:
def vacuum_to_air(wave_vac):
    """Convert vacuum wavelengths (MaNGA) to air (Lick/Balogh definitions).
    Source: Morton 1991, ApJS 77, 119 (same formula as IDL `vactoair`)."""
    s2 = (1e4 / wave_vac) ** 2
    n = 1.0 + 2.735182e-4 + 131.4182 / wave_vac**2 + 2.76249e8 / wave_vac**4
    return wave_vac / n


def band_is_covered(wave, band):
    """True if the pixels inside `band` cover at least MIN_COVERAGE of it.
    Needed because masked regions were removed from the spectra (gaps)."""
    inside = (wave >= band[0]) & (wave <= band[1])
    if inside.sum() < 2:
        return False
    pixel_size = np.median(np.diff(wave))
    return inside.sum() * pixel_size >= MIN_COVERAGE * (band[1] - band[0])


def mean_flux(wave, flux, band):
    """Average flux inside a wavelength band."""
    inside = (wave >= band[0]) & (wave <= band[1])
    return np.mean(flux[inside])


def equivalent_width(wave, flux, blue, feature, red, emission=False):
    """Lick-style equivalent width in Angstrom.
       1) mean flux in the blue and red continuum bands
       2) straight line between them = pseudo-continuum
       3) EW = integral of (1 - F / F_cont) over the feature band
       Absorption -> positive.  For emission lines we flip the sign, so
       stronger emission -> larger positive number."""
    if not all(band_is_covered(wave, b) for b in (blue, feature, red)):
        return np.nan

    # Pseudo-continuum: line through (centre_blue, F_blue) and (centre_red, F_red)
    x_b, y_b = np.mean(blue), mean_flux(wave, flux, blue)
    x_r, y_r = np.mean(red),  mean_flux(wave, flux, red)

    inside = (wave >= feature[0]) & (wave <= feature[1])
    w = wave[inside]
    continuum = y_b + (y_r - y_b) * (w - x_b) / (x_r - x_b)
    if np.any(continuum <= 0):
        return np.nan

    ew = np.trapz(1.0 - flux[inside] / continuum, w)
    return -ew if emission else ew


def d4000n(wave, flux):
    """Narrow 4000 A break = <F_nu>(red) / <F_nu>(blue)  (Balogh et al. 1999).
    F_nu ∝ F_lambda * lambda^2."""
    if not (band_is_covered(wave, D4000_BLUE) and band_is_covered(wave, D4000_RED)):
        return np.nan
    f_nu = flux * wave**2
    blue = mean_flux(wave, f_nu, D4000_BLUE)
    red  = mean_flux(wave, f_nu, D4000_RED)
    return red / blue if blue > 0 else np.nan


3. Feature extraction for ONE galaxy

In [6]:
def extract_features(wave_obs, flux, z):
    """Return a dict {feature_name: value} for one spectrum."""
    nan_result = {name: np.nan for name in FEATURE_NAMES}
    if not np.isfinite(z) or z < 0:          # missing redshift (e.g. -9999)
        return nan_result

    # Keep only valid pixels
    good = np.isfinite(wave_obs) & np.isfinite(flux)
    wave_obs, flux = wave_obs[good], flux[good]

    # --- Step 1: observed -> REST-FRAME wavelength ---
    wave_rest = wave_obs / (1.0 + z)

    # --- Step 2: vacuum -> air (to match the published band definitions) ---
    wave = vacuum_to_air(wave_rest)

    f = {"D4000n": d4000n(wave, flux)}

    # Absorption indices
    for name, (blue, feat, red) in LICK_BANDS.items():
        f[name] = equivalent_width(wave, flux, blue, feat, red)

    # Combined metallicity indicator [MgFe]'  (Thomas, Maraston & Bender 2003, MNRAS 339, 897)
    fe = 0.72 * f["Fe5270"] + 0.28 * f["Fe5335"]
    f["MgFe_prime"] = np.sqrt(f["Mgb"] * fe) if (f["Mgb"] > 0 and fe > 0) else np.nan

    # Emission lines
    for name, (blue, feat, red) in EMISSION_BANDS.items():
        f[name] = equivalent_width(wave, flux, blue, feat, red, emission=True)

    # log([NII]/Hα) from equivalent widths (only when both lines are in emission)
    if f["EW_Ha"] > 0 and f["EW_NII"] > 0:
        f["log_N2Ha"] = np.log10(f["EW_NII"] / f["EW_Ha"])
    else:
        f["log_N2Ha"] = np.nan

    return f

4. Load morphology table and build name -> redshift lookup

In [9]:
morph = Table.read(MORPH_FILE)
morph["name"] = [str(n).strip() for n in morph["name"]]
redshift = dict(zip(morph["name"], np.array(morph["nsa_z"], dtype=float)))

5. Loop over all spectra

In [ ]:
features_by_name = {}
files = sorted(glob.glob(os.path.join(SPECTRA_DIR, "*.fits.gz")))

for path in tqdm(files, desc="Extracting features"):
    # File name -> galaxy name, e.g. "manga-10001-12701-1D.fits.gz" -> "manga-10001-12701"
    match = re.match(r"(manga-\d+-\d+)", os.path.basename(path))
    if match is None:
        continue
    name = match.group(1)
    if name not in redshift:
        continue

    try:
        spec = Table.read(path, hdu=1)       # table extension (no image)
        wave = np.array(spec["WAVE"], dtype=float)
        flux = np.array(spec["FLUX_1RE"], dtype=float)
        features_by_name[name] = extract_features(wave, flux, redshift[name])
    except Exception as err:
        print(f"Skipping {name}: {err}")

print(f"Processed {len(features_by_name)} of {len(files)} files")

6. Add features as new columns and save

In [ ]:
for feature in FEATURE_NAMES:
    morph[feature] = [features_by_name.get(n, {}).get(feature, np.nan)
                      for n in morph["name"]]

morph.write(OUTPUT_FILE, overwrite=True)
print("Saved:", OUTPUT_FILE)

# Quick look: median of each feature per morphological type
df = morph.to_pandas()
print(df.groupby("Type")[["D4000n", "HdA", "Mgb", "EW_Ha"]].median().round(2))